# 5.10 · LightGBM

> **课程定位 / Where this fits**
> XGBoost(5.9)精准但在大数据上仍偏慢。微软的 LightGBM 用三件法宝——**直方图分箱、按叶生长(leaf-wise)、GOSS + EFB**——把训练速度和内存大幅优化, 精度不降反常更高。**大数据表格任务的默认首选**。
> LightGBM speeds up boosting via histogram binning, leaf-wise growth, and GOSS+EFB. The default for large tabular data.

> 💡 **面试相关 / Interview-relevant**
> - "LightGBM 为什么比 XGBoost 快" ★★★★★（直方图 + leaf-wise + GOSS/EFB）
> - "leaf-wise vs level-wise 生长的区别和风险" ★★★★★
> - "GOSS 是什么 / EFB 是什么" ★★★★
> - "num_leaves 和 max_depth 的关系" ★★★★
> - "LightGBM 怎么处理类别特征" ★★★★

---

## 学习目标 / Learning Objectives
1. **直方图分箱**为何加速(连续→离散桶)。
2. **leaf-wise vs level-wise** 生长 + 过拟合风险。
3. **GOSS**(基于梯度采样)与 **EFB**(互斥特征捆绑)思想。
4. `num_leaves` 主导调参 + 原生类别特征。
5. 与 XGBoost 对比(速度/精度)。

## 目录 / TOC
1. [三件法宝 ⭐](#1)
2. [leaf-wise vs level-wise ⭐](#2)
3. [💰 数据 + LightGBM vs XGBoost](#3)
4. [num_leaves 调参 ⭐](#4)
5. [原生类别特征](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 三件法宝 ⭐ / Three Key Tricks

**(1) 直方图分箱 (Histogram)**: 把每个连续特征预先分成约 255 个**桶**。找分裂时只需遍历桶(O(#bins))而非每个排序值(O(n))。内存小、速度快。XGBoost 后来也加了 `hist` 模式。

**(2) GOSS (Gradient-based One-Side Sampling)**: 梯度大的样本(还没学好)全保留, 梯度小的(已学好)随机下采样并加权补偿。**用更少样本估同样的分裂增益**, 加速且几乎不掉精度。

**(3) EFB (Exclusive Feature Bundling)**: 高维稀疏数据(如 one-hot)里很多特征**几乎不同时非零**(互斥), 把它们**捆成一个特征**, 大幅降维。

合起来: Light = 又轻又快。


<a id="2"></a>
## 2. leaf-wise vs level-wise ⭐ / Growth Strategy

树怎么长出来, 两种策略:
- **level-wise(按层)**: XGBoost 默认。每层所有节点一起分裂, 树平衡。
- **leaf-wise(按叶)**: LightGBM 默认。每次只分裂**当前增益最大的那个叶子**, 不管在哪层。

**leaf-wise 收益**: 同样叶子数下损失降得更多(更"贪")→ 通常精度更高、更快收敛。
**风险**: 容易长出又深又不平衡的树 → **过拟合**。所以 LightGBM 靠 **`num_leaves`** 而非 `max_depth` 控制复杂度(这是关键差异)。


<a id="3"></a>
## 3. 数据 + LightGBM vs XGBoost / Head-to-head

仍用 5.8 的**合成 Adult Income**(同生成器), 直接对比两者速度与精度。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid")
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__)

def make_income(n=60000, seed=0):
    rng = np.random.default_rng(seed)
    age = rng.integers(18, 70, n); edu_years = rng.integers(6, 21, n)
    hours = rng.normal(40, 10, n).clip(10, 80)
    capital_gain = (rng.random(n) < 0.15) * rng.exponential(5000, n)
    logit = (-9 + 0.04*age - 0.0004*(age-45)**2 + 0.25*edu_years + 0.02*hours
             + 0.0002*np.sqrt(capital_gain)*edu_years*0.3 + rng.normal(0, 0.5, n))
    y = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
    X = pd.DataFrame({"age": age, "edu_years": edu_years, "hours": hours, "capital_gain": capital_gain.round(0)})
    return X, y

X, y = make_income()    # 放大到 6 万行, 让速度差异显现
print(f"合成 Adult Income: {X.shape}, 高收入率 {y.mean():.0%}")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


In [ ]:
def timed(model, name):
    t = time.perf_counter(); model.fit(X_tr, y_tr); dt = time.perf_counter()-t
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
    print(f"{name:<12} 训练 {dt:5.2f}s   test AUC {auc:.4f}")
    return dt, auc

timed(xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                        eval_metric="auc", random_state=0), "XGBoost")
timed(lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                         random_state=0, verbose=-1), "LightGBM")
print("\n注: 两者精度相当。速度上, 现代 XGBoost 默认已用直方图(hist), 在这种")
print("    '行多但只有 4 个数值特征'的窄表上, XGBoost 甚至可能更快——LightGBM 的")
print("    GOSS/EFB/leaf-wise 优势要在'特征多、维度高(尤其稀疏/类别多)'时才明显拉开。")


<a id="4"></a>
## 4. num_leaves 调参 ⭐ / Tuning num_leaves

LightGBM 用 `num_leaves` 控复杂度。理论上一棵深度 $k$ 的平衡树有 $2^k$ 叶; 但 leaf-wise 会长不平衡, 所以**别把 num_leaves 设到 $2^{\text{max\_depth}}$ 那么大**, 否则过拟合。经验: `num_leaves` ≈ $2^{\text{max\_depth}} \times 0.6$ 上下, 并配 `min_child_samples`。


In [ ]:
leaves = [7, 15, 31, 63, 127, 255]
tr_auc, te_auc = [], []
for nl in leaves:
    m = lgb.LGBMClassifier(n_estimators=200, num_leaves=nl, learning_rate=0.05,
                           random_state=0, verbose=-1).fit(X_tr, y_tr)
    tr_auc.append(roc_auc_score(y_tr, m.predict_proba(X_tr)[:,1]))
    te_auc.append(roc_auc_score(y_te, m.predict_proba(X_te)[:,1]))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(leaves, tr_auc, "o-", label="训练 AUC")
ax.plot(leaves, te_auc, "s-", label="测试 AUC")
ax.set_xlabel("num_leaves"); ax.set_ylabel("AUC"); ax.legend()
ax.set_title("num_leaves↑: 训练AUC一路涨, 测试AUC见顶后过拟合(train-test 裂口扩大)")
plt.tight_layout(); plt.show()
print(f"训练与测试 AUC 裂口随 num_leaves 增大:",
      [f"{t-v:.3f}" for t,v in zip(tr_auc, te_auc)])


<a id="5"></a>
## 5. 原生类别特征 / Native Categorical Features

LightGBM 能**直接吃类别特征**(无需 one-hot, 3.5), 用一种基于直方图的最优分割(Fisher 1958)。指定 `categorical_feature` 即可——比 one-hot 更快更准, 尤其高基数类别。


In [ ]:
# 加一个类别特征 occupation / add a categorical feature
rng = np.random.default_rng(2)
occ_tr = pd.Categorical(rng.choice(["tech","sales","admin","service","mgmt"], len(X_tr)))
occ_te = pd.Categorical(rng.choice(["tech","sales","admin","service","mgmt"], len(X_te)))
Xc_tr = X_tr.assign(occupation=occ_tr)
Xc_te = X_te.assign(occupation=occ_te)

m = lgb.LGBMClassifier(n_estimators=200, num_leaves=31, random_state=0, verbose=-1)
m.fit(Xc_tr, y_tr, categorical_feature=["occupation"])
print(f"含原生类别特征 test AUC: {roc_auc_score(y_te, m.predict_proba(Xc_te)[:,1]):.4f}")
print("LightGBM 直接处理类别列(pandas category dtype), 免去 one-hot")
print("(本例 occupation 是随机噪声, 故 AUC 基本不变; 真实类别特征会有贡献)")


<a id="6"></a>
## 6. 小结 / Summary

```
LightGBM = 直方图分箱 + leaf-wise 生长 + GOSS + EFB → 快且省内存
直方图: 连续特征分~255桶, 分裂遍历桶 O(bins) 而非 O(n)
GOSS: 大梯度样本全留, 小梯度下采样加权 → 少样本估增益
EFB: 互斥稀疏特征捆绑降维
leaf-wise: 每次分增益最大的叶子, 更准更快但易过拟合 → 用 num_leaves 控制
原生类别特征, 无需 one-hot
```

### 💡 面试速查
1. **比 XGBoost 快**: 直方图 + leaf-wise + GOSS(梯度采样) + EFB(特征捆绑)
2. **leaf-wise(按叶最大增益) vs level-wise(按层)**: 更准但易过拟合
3. 用 **num_leaves**(而非 max_depth)控复杂度, 别设到 2^depth 满值
4. **GOSS**: 保大梯度样本; **EFB**: 捆互斥特征
5. **原生类别特征**, 省掉 one-hot

### 下一节
**5.11 CatBoost**——Yandex 的 boosting。专治类别特征(有序目标编码防泄漏)+ 对称树, 默认参数就很强, 类别特征多时首选。
